# Elastic Properties & Rock Strength — Validation via `unit_converter`

This notebook exercises **every public function** in `geomechpy/elastic_properties.py`
and `geomechpy/rock_strength.py`, using `geomechpy/unit_converter.py` as the unit
bridge between them, and prints the output of each step so the calculations can be
checked by eye and by assertion.

**Workflow validated**

1. `UnitConverter` self-consistency (pressure, mud-weight/density, length, acoustic).
2. Dynamic elastic properties from **slowness** and from **velocity** — cross-checked
   through `UnitConverter.convert_acoustic`.
3. All **15 pairwise** elastic-modulus conversions reproduce one self-consistent
   reference state, and are shown to be **unit-independent** (GPa vs Mpsi) via the converter.
4. Rock strength (UCS, tensile strength, friction angle) computed from the dynamic
   moduli, with `UnitConverter.convert_pressure` bridging Pa → Mpsi (the Plumb input unit).
5. Array (log-table) variants of every function, presented as a depth table.
6. Machine-checked assertions — the final cell prints `ALL CHECKS PASSED`.

> Units convention of the library: the pairwise converters are unit-agnostic (outputs
> share the input pressure unit); `convert_dynamic_*` return **Pa**; the Plumb UCS
> correlation expects Young's modulus in **Mpsi** and returns **psi**; Lal friction
> angle takes compressional slowness in **µs/ft**.


## 0 — Setup

In [1]:

import sys, math
from pathlib import Path

import numpy as np
import pandas as pd

# Make the repo's geomechpy importable whether this runs from validation/ or the repo root.
here = Path.cwd()
for candidate in (here, here.parent, here.parent.parent):
    if (candidate / "geomechpy" / "elastic_properties.py").exists():
        sys.path.insert(0, str(candidate))
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the geomechpy package from " + str(here))

from geomechpy.elastic_properties import ElasticProperties, ElasticPropertiesConverter
from geomechpy.rock_strength import RockStrengthPropertiesConverter
from geomechpy.unit_converter import UnitConverter

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print("geomechpy loaded from:", REPO_ROOT)
print("pandas", pd.__version__, "| numpy", np.__version__)

FIELDS = ["bulk_modulus", "youngs_modulus", "lame_parameter",
          "shear_modulus", "poissons_ratio", "p_wave_modulus"]

def ep_to_dict(ep: ElasticProperties) -> dict:
    # ElasticProperties -> plain dict (for tabulation)
    return {f: getattr(ep, f) for f in FIELDS}


geomechpy loaded from: /home/user/GeomechPy_smolrun
pandas 3.0.5 | numpy 2.5.3


## 1 — `UnitConverter` self-consistency

Before relying on the converter as a bridge, confirm it against first-principles
constants and round-trips. `assert` guards every claim.

In [2]:

checks = []

# -- pressure: anchor constants + round trip
one_psi_in_pa = UnitConverter.convert_pressure(1.0, "psi", "pa")
one_mpsi_in_psi = UnitConverter.convert_pressure(1.0, "mpsi", "psi")
gpa_roundtrip = UnitConverter.convert_pressure(
    UnitConverter.convert_pressure(42.0, "gpa", "psi"), "psi", "gpa")
assert math.isclose(one_psi_in_pa, 6894.757293168, rel_tol=1e-12)
assert math.isclose(one_mpsi_in_psi, 1e6, rel_tol=1e-12)
assert math.isclose(gpa_roundtrip, 42.0, rel_tol=1e-12)
checks += [("1 psi -> Pa", one_psi_in_pa, 6894.757293168),
           ("1 Mpsi -> psi", one_mpsi_in_psi, 1e6),
           ("42 GPa -> psi -> GPa", gpa_roundtrip, 42.0)]

# -- mud weight / density: 1 g/cm3 == 1 sg, ppg anchor
assert math.isclose(UnitConverter.convert_equivalent_mud_weight(1.0, "g/cm3", "sg"), 1.0, rel_tol=1e-12)
ppg_of_1sg = UnitConverter.convert_equivalent_mud_weight(1.0, "sg", "ppg")
assert math.isclose(ppg_of_1sg, 8.345404, rel_tol=1e-9)
checks += [("1 g/cm3 -> sg", 1.0, 1.0), ("1 sg -> ppg", ppg_of_1sg, 8.345404)]

# -- length: 1 ft == 0.3048 m, round trip
assert math.isclose(UnitConverter.convert_length(1.0, "ft", "m"), 0.3048, rel_tol=1e-12)
assert math.isclose(UnitConverter.convert_length(UnitConverter.convert_length(500.0, "m", "ft"), "ft", "m"), 500.0, rel_tol=1e-12)
checks += [("1 ft -> m", UnitConverter.convert_length(1.0, "ft", "m"), 0.3048)]

# -- acoustic: slowness<->velocity reciprocal, and us/ft -> m/s equals 304800/slowness
v_from_dt = UnitConverter.convert_acoustic(70.0, "us/ft", "m/s")
assert math.isclose(v_from_dt, 304800.0 / 70.0, rel_tol=1e-9)
dt_back = UnitConverter.convert_acoustic(v_from_dt, "m/s", "us/ft")
assert math.isclose(dt_back, 70.0, rel_tol=1e-9)
checks += [("70 us/ft -> m/s", v_from_dt, 304800.0 / 70.0),
           ("...-> us/ft (round trip)", dt_back, 70.0)]

pd.DataFrame(checks, columns=["check", "converter_result", "expected"])


,check,converter_result,expected
0,1 psi -> Pa,"6,894.7573","6,894.7573"
1,1 Mpsi -> psi,"1,000,000.0000","1,000,000.0000"
2,42 GPa -> psi -> GPa,42.0000,42.0000
3,1 g/cm3 -> sg,1.0000,1.0000
4,1 sg -> ppg,8.3454,8.3454
5,1 ft -> m,0.3048,0.3048
6,70 us/ft -> m/s,"4,354.2857","4,354.2857"
7,...-> us/ft (round trip),70.0000,70.0000


## 2 — Dynamic elastic properties from slowness and velocity

Realistic shaly-sandstone log reading: **DTCO = 70 µs/ft**, **DTSM = 120 µs/ft**,
**RHOB = 2350 kg/m³**. The slowness route and the velocity route must agree, with
`UnitConverter.convert_acoustic` providing the µs/ft → m/s conversion.

In [3]:

DTCO_US_FT = 70.0     # compressional slowness
DTSM_US_FT = 120.0    # shear slowness
RHOB_KGM3  = 2350.0   # bulk density

# Route A: directly from slowness (returns Pa)
ep_slow = ElasticPropertiesConverter.convert_dynamic_elastic_properties_from_slowness(
    p_wave_slowness=DTCO_US_FT, s_wave_slowness=DTSM_US_FT, density=RHOB_KGM3)

# Route B: convert slowness -> velocity (m/s) with the UnitConverter, then from velocity
vp_ms = UnitConverter.convert_acoustic(DTCO_US_FT, "us/ft", "m/s")
vs_ms = UnitConverter.convert_acoustic(DTSM_US_FT, "us/ft", "m/s")
ep_vel = ElasticPropertiesConverter.convert_dynamic_elastic_properties_from_velocity(
    p_wave_velocity=vp_ms, s_wave_velocity=vs_ms, density=RHOB_KGM3)

print(f"Vp = {vp_ms:,.2f} m/s   Vs = {vs_ms:,.2f} m/s   (from UnitConverter.convert_acoustic)")

# The two routes must be identical
for f in FIELDS:
    assert math.isclose(getattr(ep_slow, f), getattr(ep_vel, f), rel_tol=1e-9), f
print("Slowness route and velocity route agree to 1e-9 -> OK")

# Present the dynamic moduli in Pa, GPa, psi and Mpsi via UnitConverter.convert_pressure
rows = []
for f in FIELDS:
    pa = getattr(ep_slow, f)
    if f == "poissons_ratio":
        rows.append([f, pa, np.nan, np.nan, np.nan])
    else:
        rows.append([f, pa,
                     UnitConverter.convert_pressure(pa, "pa", "gpa"),
                     UnitConverter.convert_pressure(pa, "pa", "psi"),
                     UnitConverter.convert_pressure(pa, "pa", "mpsi")])
dyn_df = pd.DataFrame(rows, columns=["property", "Pa", "GPa", "psi", "Mpsi"])
dyn_df


Vp = 4,354.29 m/s   Vs = 2,540.00 m/s   (from UnitConverter.convert_acoustic)
Slowness route and velocity route agree to 1e-9 -> OK


,property,Pa,GPa,psi,Mpsi
0,bulk_modulus,"24,340,526,258.5034",24.3405,"3,530,294.8637",3.5303
1,youngs_modulus,"37,663,761,684.2105",37.6638,"5,462,666.7891",5.4627
2,lame_parameter,"14,233,019,591.8367",14.2330,"2,064,324.9627",2.0643
3,shear_modulus,"15,161,260,000.0000",15.1613,"2,198,954.8515",2.1990
4,poissons_ratio,0.2421,NaN,NaN,NaN
5,p_wave_modulus,"44,555,539,591.8367",44.5555,"6,462,234.6657",6.4622


## 3 — All 15 pairwise conversions reproduce one reference state

Reference isotropic solid (chosen for round numbers), in **GPa**:

| K | E | λ | G | ν | M |
|---|---|---|---|---|---|
| 16 | 28.8 | 8 | 12 | 0.2 | 32 |

Every `convert_from_<x>_and_<y>` is fed the matching pair and must return the full
six-value state. The table lists the maximum absolute error of each method against
the reference.

In [4]:

REF = dict(bulk_modulus=16.0, youngs_modulus=28.8, lame_parameter=8.0,
           shear_modulus=12.0, poissons_ratio=0.2, p_wave_modulus=32.0)

C = ElasticPropertiesConverter
# (label, method, (arg1_field, arg2_field)) — args pulled from REF
PAIR_METHODS = [
    ("bulk & youngs",     C.convert_from_bulk_and_youngs,     ("bulk_modulus", "youngs_modulus")),
    ("bulk & lame",       C.convert_from_bulk_and_lame,       ("bulk_modulus", "lame_parameter")),
    ("bulk & shear",      C.convert_from_bulk_and_shear,      ("bulk_modulus", "shear_modulus")),
    ("bulk & poissons",   C.convert_from_bulk_and_poissons,   ("bulk_modulus", "poissons_ratio")),
    ("bulk & p_wave",     C.convert_from_bulk_and_p_wave,     ("bulk_modulus", "p_wave_modulus")),
    ("youngs & lame",     C.convert_from_youngs_and_lame,     ("youngs_modulus", "lame_parameter")),
    ("youngs & shear",    C.convert_from_youngs_and_shear,    ("youngs_modulus", "shear_modulus")),
    ("youngs & poissons", C.convert_from_youngs_and_poissons, ("youngs_modulus", "poissons_ratio")),
    ("youngs & p_wave",   C.convert_from_youngs_and_p_wave,   ("youngs_modulus", "p_wave_modulus")),
    ("lame & shear",      C.convert_from_lame_and_shear,      ("lame_parameter", "shear_modulus")),
    ("lame & poissons",   C.convert_from_lame_and_poissons,   ("lame_parameter", "poissons_ratio")),
    ("lame & p_wave",     C.convert_from_lame_and_p_wave,     ("lame_parameter", "p_wave_modulus")),
    ("shear & poissons",  C.convert_from_shear_and_poissons,  ("shear_modulus", "poissons_ratio")),
    ("shear & p_wave",    C.convert_from_shear_and_p_wave,    ("shear_modulus", "p_wave_modulus")),
    ("poissons & p_wave", C.convert_from_poissons_and_p_wave, ("poissons_ratio", "p_wave_modulus")),
]

records, max_err = [], 0.0
for label, method, (fa, fb) in PAIR_METHODS:
    ep = method(REF[fa], REF[fb])
    d = ep_to_dict(ep)
    err = max(abs(d[f] - REF[f]) for f in FIELDS)
    max_err = max(max_err, err)
    assert err < 1e-9, (label, err)
    records.append({"method": label, **d, "max_abs_err": err})

print(f"All 15 pairwise methods reproduce the reference (worst error = {max_err:.2e})")
pd.DataFrame(records).set_index("method")


All 15 pairwise methods reproduce the reference (worst error = 7.11e-15)


,bulk_modulus,youngs_modulus,lame_parameter,shear_modulus,poissons_ratio,p_wave_modulus,max_abs_err
method,,,,,,,
bulk & youngs,16.0000,28.8000,8.0000,12.0000,0.2000,32.0000,0.0000
bulk & lame,16.0000,28.8000,8.0000,12.0000,0.2000,32.0000,0.0000
bulk & shear,16.0000,28.8000,8.0000,12.0000,0.2000,32.0000,0.0000
bulk & poissons,16.0000,28.8000,8.0000,12.0000,0.2000,32.0000,0.0000
bulk & p_wave,16.0000,28.8000,8.0000,12.0000,0.2000,32.0000,0.0000
youngs & lame,16.0000,28.8000,8.0000,12.0000,0.2000,32.0000,0.0000
youngs & shear,16.0000,28.8000,8.0000,12.0000,0.2000,32.0000,0.0000
youngs & poissons,16.0000,28.8000,8.0000,12.0000,0.2000,32.0000,0.0000
youngs & p_wave,16.0000,28.8000,8.0000,12.0000,0.2000,32.0000,0.0000


### 3b — Unit independence through the converter

The library claims the pairwise outputs carry the input pressure unit. Convert the
reference pair to **Mpsi** with `UnitConverter`, run the conversion there, convert the
outputs back to **GPa**, and confirm they match the GPa reference.

In [5]:

# Use bulk & shear as the exemplar
K_mpsi = UnitConverter.convert_pressure(REF["bulk_modulus"], "gpa", "mpsi")
G_mpsi = UnitConverter.convert_pressure(REF["shear_modulus"], "gpa", "mpsi")
ep_mpsi = ElasticPropertiesConverter.convert_from_bulk_and_shear(K_mpsi, G_mpsi)

back_to_gpa = {}
for f in FIELDS:
    val = getattr(ep_mpsi, f)
    back_to_gpa[f] = val if f == "poissons_ratio" else UnitConverter.convert_pressure(val, "mpsi", "gpa")
    assert math.isclose(back_to_gpa[f], REF[f], rel_tol=1e-9), f

print("Computed in Mpsi, converted back to GPa, matches the GPa reference -> OK")
pd.DataFrame({"reference_GPa": REF,
              "computed_in_Mpsi": {f: getattr(ep_mpsi, f) for f in FIELDS},
              "back_to_GPa": back_to_gpa})


Computed in Mpsi, converted back to GPa, matches the GPa reference -> OK


,reference_GPa,computed_in_Mpsi,back_to_GPa
bulk_modulus,16.0000,2.3206,16.0000
youngs_modulus,28.8000,4.1771,28.8000
lame_parameter,8.0000,1.1603,8.0000
shear_modulus,12.0000,1.7405,12.0000
poissons_ratio,0.2000,0.2000,0.2000
p_wave_modulus,32.0000,4.6412,32.0000


## 4 — Rock strength from the dynamic moduli (converter as the bridge)

`RockStrengthPropertiesConverter.convert_yme_sta_to_ucs_plumb` expects Young's modulus
in **Mpsi**; the dynamic calculation produced it in **Pa**. `UnitConverter.convert_pressure`
bridges the two. Friction angle comes straight from DTCO (µs/ft).

In [6]:

RS = RockStrengthPropertiesConverter

# Young's modulus from the Section-2 dynamic result (Pa) -> Mpsi via the converter
youngs_pa = ep_slow.youngs_modulus
youngs_mpsi = UnitConverter.convert_pressure(youngs_pa, "pa", "mpsi")

ucs_psi = RS.convert_yme_sta_to_ucs_plumb(youngs_mpsi)          # Plumb generic
tstr_psi = RS.convert_ucs_to_tstr(ucs_psi)                      # default 0.15 * UCS
fang_deg = RS.convert_friction_angle_lal(DTCO_US_FT)            # Lal, from compressional slowness

# Arithmetic assertions against the documented formulas
assert math.isclose(ucs_psi, 0.210306770614015 * youngs_mpsi, rel_tol=1e-12)
assert math.isclose(tstr_psi, 0.15 * ucs_psi, rel_tol=1e-12)
assert math.isclose(fang_deg,
                    (180.0 / 3.141592) * math.asin((304800 - 1000 * DTCO_US_FT) / (304800 + 1000 * DTCO_US_FT)),
                    rel_tol=1e-12)

summary = pd.DataFrame([
    ["Young's modulus (dynamic)", youngs_pa, "Pa"],
    ["Young's modulus (dynamic)", youngs_mpsi, "Mpsi  (via UnitConverter)"],
    ["UCS  (Plumb generic)", ucs_psi, "psi"],
    ["UCS  (Plumb generic)", UnitConverter.convert_pressure(ucs_psi, "psi", "mpa"), "MPa  (via UnitConverter)"],
    ["Tensile strength (0.15*UCS)", tstr_psi, "psi"],
    ["Friction angle (Lal)", fang_deg, "deg"],
], columns=["quantity", "value", "unit"])
summary


,quantity,value,unit
0,Young's modulus (dynamic),"37,663,761,684.2105",Pa
1,Young's modulus (dynamic),5.4627,Mpsi (via UnitConverter)
2,UCS (Plumb generic),1.1488,psi
3,UCS (Plumb generic),0.0079,MPa (via UnitConverter)
4,Tensile strength (0.15*UCS),0.1723,psi
5,Friction angle (Lal),38.7900,deg


## 5 — Array / log-table variants

A small synthetic log (5 depths) drives every `*_array` function in all three modules,
with `UnitConverter.*_array` handling the depth, density and slowness columns. The
result is one tidy depth table.

In [7]:

depth_ft   = [6000.0, 6500.0, 7000.0, 7500.0, 8000.0]
dtco_us_ft = [95.0, 88.0, 80.0, 72.0, 66.0]
dtsm_us_ft = [170.0, 158.0, 145.0, 130.0, 120.0]
rhob_kgm3  = [2300.0, 2330.0, 2360.0, 2400.0, 2430.0]

# --- UnitConverter array helpers ---
depth_m     = UnitConverter.convert_length_array(depth_ft, "ft", "m")
rhob_sg     = UnitConverter.convert_equivalent_mud_weight_array(rhob_kgm3, "kg/m3", "sg")
vp_ms_arr   = UnitConverter.convert_acoustic_array(dtco_us_ft, "us/ft", "m/s")
vs_ms_arr   = UnitConverter.convert_acoustic_array(dtsm_us_ft, "us/ft", "m/s")

# --- elastic_properties array variants: slowness route and velocity route ---
ep_slow_arr = ElasticPropertiesConverter.convert_dynamic_elastic_properties_from_slowness_array(
    dtco_us_ft, dtsm_us_ft, rhob_kgm3)
ep_vel_arr = ElasticPropertiesConverter.convert_dynamic_elastic_properties_from_velocity_array(
    vp_ms_arr, vs_ms_arr, rhob_kgm3)
for a, b in zip(ep_slow_arr, ep_vel_arr):
    for f in FIELDS:
        assert math.isclose(getattr(a, f), getattr(b, f), rel_tol=1e-9)

youngs_pa_arr   = [ep.youngs_modulus for ep in ep_slow_arr]
youngs_mpsi_arr = UnitConverter.convert_pressure_array(youngs_pa_arr, "pa", "mpsi")

# --- rock_strength array variants ---
ucs_arr  = RockStrengthPropertiesConverter.convert_yme_sta_to_ucs_plumb_array(youngs_mpsi_arr)
tstr_arr = RockStrengthPropertiesConverter.convert_ucs_to_tstr_array(ucs_arr)
fang_arr = RockStrengthPropertiesConverter.convert_friction_angle_lal_array(dtco_us_ft)

log = pd.DataFrame({
    "depth_ft": depth_ft,
    "depth_m": depth_m,
    "DTCO_us/ft": dtco_us_ft,
    "DTSM_us/ft": dtsm_us_ft,
    "RHOB_kg/m3": rhob_kgm3,
    "RHOB_sg": rhob_sg,
    "Vp_m/s": vp_ms_arr,
    "Vs_m/s": vs_ms_arr,
    "E_dyn_Mpsi": youngs_mpsi_arr,
    "nu_dyn": [ep.poissons_ratio for ep in ep_slow_arr],
    "UCS_psi": ucs_arr,
    "TSTR_psi": tstr_arr,
    "FANG_deg": fang_arr,
})
log


,depth_ft,depth_m,DTCO_us/ft,DTSM_us/ft,RHOB_kg/m3,RHOB_sg,Vp_m/s,Vs_m/s,E_dyn_Mpsi,nu_dyn,UCS_psi,TSTR_psi,FANG_deg
0,"6,000.0000","1,828.8000",95.0000,170.0000,"2,300.0000",2.3000,"3,208.4211","1,792.9412",2.7301,0.2730,0.5742,0.0861,31.6523
1,"6,500.0000","1,981.2000",88.0000,158.0000,"2,330.0000",2.3300,"3,463.6364","1,929.1139",3.2073,0.2751,0.6745,0.1012,33.4999
2,"7,000.0000","2,133.6000",80.0000,145.0000,"2,360.0000",2.3600,"3,810.0000","2,102.0690",3.8755,0.2812,0.8151,0.1223,35.7465
3,"7,500.0000","2,286.0000",72.0000,130.0000,"2,400.0000",2.4000,"4,233.3333","2,344.6154",4.8939,0.2788,1.0292,0.1544,38.1582
4,"8,000.0000","2,438.4000",66.0000,120.0000,"2,430.0000",2.4300,"4,618.1818","2,540.0000",5.8353,0.2832,1.2272,0.1841,40.0917


### 5b — Remaining pairwise *array* variants

The 15 `convert_from_*_array` pair methods are each driven with two-element inputs
(built from the Section-3 reference, repeated) and checked against the reference, so
every array entry point is exercised.

In [8]:

PAIR_ARRAY_METHODS = [
    ("bulk & youngs",     C.convert_from_bulk_and_youngs_array,     ("bulk_modulus", "youngs_modulus")),
    ("bulk & lame",       C.convert_from_bulk_and_lame_array,       ("bulk_modulus", "lame_parameter")),
    ("bulk & shear",      C.convert_from_bulk_and_shear_array,      ("bulk_modulus", "shear_modulus")),
    ("bulk & poissons",   C.convert_from_bulk_and_poissons_array,   ("bulk_modulus", "poissons_ratio")),
    ("bulk & p_wave",     C.convert_from_bulk_and_p_wave_array,     ("bulk_modulus", "p_wave_modulus")),
    ("youngs & lame",     C.convert_from_youngs_and_lame_array,     ("youngs_modulus", "lame_parameter")),
    ("youngs & shear",    C.convert_from_youngs_and_shear_array,    ("youngs_modulus", "shear_modulus")),
    ("youngs & poissons", C.convert_from_youngs_and_poissons_array, ("youngs_modulus", "poissons_ratio")),
    ("youngs & p_wave",   C.convert_from_youngs_and_p_wave_array,   ("youngs_modulus", "p_wave_modulus")),
    ("lame & shear",      C.convert_from_lame_and_shear_array,      ("lame_parameter", "shear_modulus")),
    ("lame & poissons",   C.convert_from_lame_and_poissons_array,   ("lame_parameter", "poissons_ratio")),
    ("lame & p_wave",     C.convert_from_lame_and_p_wave_array,     ("lame_parameter", "p_wave_modulus")),
    ("shear & poissons",  C.convert_from_shear_and_poissons_array,  ("shear_modulus", "poissons_ratio")),
    ("shear & p_wave",    C.convert_from_shear_and_p_wave_array,    ("shear_modulus", "p_wave_modulus")),
    ("poissons & p_wave", C.convert_from_poissons_and_p_wave_array, ("poissons_ratio", "p_wave_modulus")),
]

n_entry, worst = 0, 0.0
for label, method, (fa, fb) in PAIR_ARRAY_METHODS:
    out = method([REF[fa], REF[fa]], [REF[fb], REF[fb]])
    assert len(out) == 2
    for ep in out:
        d = ep_to_dict(ep)
        worst = max(worst, max(abs(d[f] - REF[f]) for f in FIELDS))
        n_entry += 1
assert worst < 1e-9
print(f"All 15 pairwise array methods validated over {n_entry} entries (worst error = {worst:.2e}) -> OK")


All 15 pairwise array methods validated over 30 entries (worst error = 7.11e-15) -> OK


## 6 — Final verdict

In [9]:

# Manual ElasticProperties construction (the dataclass itself)
manual = ElasticProperties(bulk_modulus=16.0, youngs_modulus=28.8, lame_parameter=8.0,
                           shear_modulus=12.0, poissons_ratio=0.2, p_wave_modulus=32.0)
assert ep_to_dict(manual)["p_wave_modulus"] == 32.0

print("ElasticProperties dataclass .......... OK")
print("15 pairwise scalar converters ........ OK")
print("15 pairwise array converters ......... OK")
print("dynamic from slowness / velocity ..... OK (scalar + array, cross-checked)")
print("rock strength UCS / TSTR / FANG ...... OK (scalar + array)")
print("UnitConverter pressure/mw/length/acoustic + array ... OK")
print()
print("ALL CHECKS PASSED")


ElasticProperties dataclass .......... OK
15 pairwise scalar converters ........ OK
15 pairwise array converters ......... OK
dynamic from slowness / velocity ..... OK (scalar + array, cross-checked)
rock strength UCS / TSTR / FANG ...... OK (scalar + array)
UnitConverter pressure/mw/length/acoustic + array ... OK

ALL CHECKS PASSED
